<a href="https://colab.research.google.com/github/Naylet92/Estudio_ambiental/blob/main/San%20Gabriel/SGB_capas_entrenamiento.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Praparar datos de entrenamiento para Sierra de San Gabriel

In [1]:
# Prefijo del área
PREFIJO     = 'SGB'
# Nombre del área
NOMBRE      = 'San_Gabriel'

# Carpeta de salida en Drive
#RUTA_AOI    = '/content/drive/MyDrive/ANP/AOI/Colima'
RUTA_AOI    = '/content/drive/MyDrive/Colab Data/Naylet/AOI'

# Rutas
RUTA_GEOJSON = f'{RUTA_AOI}/aoi_{PREFIJO}.geojson'
RUTA_BBOX   = f'{RUTA_AOI}/{PREFIJO}_bbox.geojson'
RUTA_CSV    = f'{RUTA_AOI}/{PREFIJO}_coordenadas.csv'

# CRS geográfico y UTM
CRS_GEO     = 4326
CRS_UTM     = 32613

# Proyecto de Google Earth Engine
#GEE_PROJECT = 'ee-nayleths'
GEE_PROJECT = 'ee-vshalisko'

# Año
AÑO = 2020

fecha_inicio_exacto = f'{AÑO}-01-01'
fecha_fin_exacto   = f'{AÑO}-12-31'

# Escala de exportación (m)
ESCALA        = 30

# Zoom inicial del mapa
ZOOM        = 12

# Estilo del área de estudio
STYLE_AREA  = {'color': 'blue', 'fillColor': '#0000ff30', 'weight': 1.5}

# Estilo del rectángulo rojo o bounding box
STYLE_BBOX  = {'color': 'red', 'fillColor': '#00000000', 'weight': 2.5}

# Título del mapa
MAP_TITLE   = 'Área de estudio Sierra de San Gabriel'

# Carpeta de salida en Drive para imágenes GEE
#RUTA_IMAGENES = f'/content/drive/MyDrive/ANP/Landsat'
RUTA_IMAGENES = 'Colab Data NDC 2020'


In [2]:
import ee
import os
import math
import json
import geemap
import pandas as pd
import geopandas as gpd
from shapely.geometry import box
from google.colab import drive
#from IPython.display import display, HTML

# Montar Google Drive
drive.mount('/content/drive')

# Autenticar e inicializar GEE
ee.Authenticate()
ee.Initialize(project=GEE_PROJECT)

print("✓ Entorno listo")

Mounted at /content/drive
✓ Entorno listo


### Cargar rectangulo de interés

In [3]:
# Región GEE desde el GeoJSON generado
gdf = gpd.read_file(RUTA_BBOX)
geojson_str = gdf.to_crs(epsg=CRS_GEO).to_json()
region = ee.FeatureCollection(json.loads(geojson_str)).geometry()

# Recalcular centroide
gdf_bbox = gpd.read_file(RUTA_BBOX).to_crs(epsg=CRS_GEO)

# Calcular centroide en WGS84
# FIX: calcular centroide en CRS proyectado, luego volver a WGS84
#centroid_proj = gdf_bbox.to_crs(epsg=3857).dissolve().centroid.iloc[0]
#centroid = gpd.GeoSeries([centroid_proj], crs=3857).to_crs(epsg=4326).iloc[0]

# Bases de datos con LULC de referencia

1. EC JRC global map of forest types V1 (2020)
`COPERNICUS/Landcover/100m/Proba-V-C3/Global/2019`
2. Dynamic World (Derivado de Sentinel)
`GOOGLE/DYNAMICWORLD/V1`
3. Copernicus Global Land Cover Layers (2015 en adelante)
`COPERNICUS/Landcover/100m/Proba-V-C3/Global/2019`

In [4]:

ref_image_EC = ee.Image('JRC/GFC2020_subtypes/V1')

ref_image_CP = ee.Image('COPERNICUS/Landcover/100m/Proba-V-C3/Global/2019').select(
    'discrete_classification'
)

Para Dynamic Word se requiere calcular la moda del etiquetado de clases para el periodo de interes a partir de la colección

In [5]:
ref_collection_DW = (ee.ImageCollection('GOOGLE/DYNAMICWORLD/V1')
                .filterDate(fecha_inicio_exacto, fecha_fin_exacto)
                .filterBounds(region))

ref_image_DW = ref_collection_DW.first()

ref_image_DW_mode = ref_collection_DW.select('label').reduce(ee.Reducer.mode())
ref_image_DW_label_mode = ref_image_DW_mode.select('label_mode')

In [6]:
# Define list pairs of DW LULC label and color.
CLASS_NAMES = [
    'water',
    'trees',
    'grass',
    'flooded_vegetation',
    'crops',
    'shrub_and_scrub',
    'built',
    'bare',
    'snow_and_ice',
]

VIS_PALETTE = [
    '419bdf',
    '397d49',
    '88b053',
    '7a87c6',
    'e49635',
    'dfc35a',
    'c4281b',
    'a59b8f',
    'b39fe1',
]

# Create an RGB image of the label (most likely class) on [0, 1].
dw_rgb = (
    #ref_image_DW.select('label')
    ref_image_DW_mode.select('label_mode')
    .visualize(min=0, max=8, palette=VIS_PALETTE)
    .divide(255)
)

In [ ]:
#  MAPA
m = geemap.Map(height="950px", width="100%",layer_ctrl=False)
m.centerObject(region, zoom=ZOOM)

m.add_gdf(gdf_bbox, layer_name='ANP – Bounding Box', style=STYLE_BBOX)
m.add_basemap('ROADMAP')

#m.set_center(-103.5, 19.5, 11)

m.add_layer(
    dw_rgb,
    {'min': 0, 'max': 1},
    'Dynamic World V1 - label hillshade',
)

m.add_layer(ref_image_EC, {}, 'Land Cover EC')
m.add_layer(ref_image_CP, {}, 'Land Cover CP')
m

m

In [8]:
# Exportar a Google Drive
def exportar(imagen, fuente):
    if imagen is None:
        print(f"Exportación cancelada para '{fuente}': imagen no disponible.")
        return

    nombre_tarea = f'{PREFIJO}_{AÑO}_{fuente}'
    tarea = ee.batch.Export.image.toDrive(
        image          = imagen,
        description    = nombre_tarea,
        folder         = RUTA_IMAGENES,
        fileNamePrefix = nombre_tarea,
        region         = region,
        scale          = ESCALA,
        crs            = f'EPSG:{CRS_UTM}',
        maxPixels      = 1e13
    )
    tarea.start()
    print(f"Tarea iniciada: {nombre_tarea}")

exportar(ref_image_EC,  'ref_EC')
exportar(ref_image_CP,  'ref_CP')
exportar(ref_image_DW_label_mode,  'ref_DW')


Tarea iniciada: SGB_2020_ref_EC
Tarea iniciada: SGB_2020_ref_CP
Tarea iniciada: SGB_2020_ref_DW
